# Day 5 — Delta Lake: Architecture, Formats & Schema

Plan for about 2–2.5 hours in the notebook, plus any discussion. Work through the parts in order, or split early ingest sections and later practice sections across the class if needed.

Topics include Parquet vs Delta and the transaction log, CSV and JSON loads to Delta with `overwrite` / `append` / `mergeSchema`, partitioned writes, `DESCRIBE DETAIL` and `DESCRIBE HISTORY`, strict vs evolving schema, `replaceWhere`, SQL reads on `delta.` paths, data quality checks, skill-builder cells, and optional challenges.

Prerequisites: `%run ./02-Mount-Azure-Data-Lake`, and the usual course files `2010-summary.csv` and `json/2015-summary.json` on the storage account. In UC-oriented workspaces, prefer `_metadata.file_path` (or a literal path) over `input_file_name()` for file lineage.

Public Databricks training labs on Delta reads/writes, versioning, schema evolution, and optimizations are broadly aligned with this notebook; paths here use ABFS as in the rest of this course.

In [0]:
BASE_PATH = "abfss://atininput@sadbtrng19032026.dfs.core.windows.net/data/"
STUDENT_ID = "u25"  # Change to your assigned u01–u16
DAY5 = BASE_PATH + "/day5"
P_BASIC = DAY5 + "/delta/flight_summary_basic"
P_JSON = DAY5 + "/delta/flight_json"
P_PART = DAY5 + "/delta/flight_partitioned"
P_STRICT = DAY5 + "/delta/schema_strict"
P_EVOLVE = DAY5 + "/delta/schema_evolve_demo"
SOURCE_CSV = BASE_PATH + "/2010-summary.csv"
SOURCE_JSON = BASE_PATH + "/json/2015-summary.json"

# Verify source files exist
print("=== Checking Prerequisites ===")
prereqs_met = True
try:
    df_csv = spark.read.format("csv").option("header", True).option("inferSchema", True).load(SOURCE_CSV).limit(1)
    print(f"✓ Source CSV exists: {SOURCE_CSV}")
except Exception as e:
    print(f"✗ Source CSV NOT found: {SOURCE_CSV}")
    print(f"  Error: {type(e).__name__}")
    prereqs_met = False

try:
    df_json = spark.read.format("json").load(SOURCE_JSON).limit(1)
    print(f"✓ Source JSON exists: {SOURCE_JSON}")
except Exception as e:
    print(f"✗ Source JSON NOT found: {SOURCE_JSON}")
    print(f"  Error: {type(e).__name__}")
    prereqs_met = False

if not prereqs_met:
    print("\n⚠️  WARNING: Some prerequisite data files are missing.")
    print("   Delta read/write operations will fail.")

print("\n=== Path Configuration ===")
for _n, _p in [
    ("DAY5 root", DAY5),
    ("basic", P_BASIC),
    ("json delta", P_JSON),
    ("partitioned", P_PART),
    ("strict", P_STRICT),
    ("evolve", P_EVOLVE),
]:
    print(_n + ":", _p)

## Part B — Theory: Parquet-only lakes vs Delta

**Plain Parquet directories**
- No single **transaction log** → readers may see **partial writes** if a job fails mid-write.
- **No built-in ACID** semantics across concurrent writers.
- **Schema drift** is painful: files disagree silently.

**Delta Lake**
- **`_delta_log`** records commits (JSON + checkpoints).
- **ACID** table semantics at the directory level.
- **Time travel**, **CLONE**, **MERGE/UPDATE/DELETE**, **Change Data Feed** (later topics / Day 6).
- **Schema enforcement** and controlled **evolution**.

### B2 — Checklist (Delta features you will touch today)

| Feature | Where in Day 5 |
|---------|----------------|
| ACID overwrite / append | Parts C–E |
| Transaction log (history) | Part F |
| Schema enforcement | Part G |
| Schema evolution (`mergeSchema`) | Part G |
| Partitioning | Part E |
| Optimize (Databricks) | Notebook **03** |

### B3 — Inside `_delta_log` (mental model)

Each commit appends **JSON** files (and periodic **checkpoints**) under **`_delta_log/`** at the table root. Readers always see a **consistent snapshot** from the latest successful commit.

| Artifact | Role |
|----------|------|
| `*.json` commit | Add/remove files, schema, metadata, metrics |
| Checkpoints | Speed up log replay for large tables |
| Reader protocol | Uses log to decide which Parquet files are **active** |

**Takeaway:** Delta is **Parquet files + transaction log**, not “magic storage” — governance still lives in **ABFS ACLs** + **Unity Catalog** when you register tables.

## Part C — First Delta table: CSV → Delta (`overwrite`)

In [0]:
from pyspark.sql.functions import coalesce, col, current_timestamp, lit

df0 = (
    spark.read.format("csv")
    .option("header", True)
    .option("inferSchema", True)
    .load(SOURCE_CSV)
    .withColumn("ingested_at", current_timestamp())
    .withColumn("source_path", coalesce(col("_metadata.file_path"), lit(SOURCE_CSV)))
)
df0.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(P_BASIC)
print("Rows:", df0.count(), "| Path:", P_BASIC)

In [0]:
spark.read.format("delta").load(P_BASIC).printSchema()
spark.read.format("delta").load(P_BASIC).display(5, truncate=False)

## Part D — Append with `append` + extra column (`mergeSchema`)

In [0]:
from pyspark.sql.functions import coalesce, col, current_timestamp, lit, expr

append_batch = (
    spark.read.format("csv")
    .option("header", True)
    .option("inferSchema", True)
    .load(SOURCE_CSV)
    .withColumn("ingested_at", current_timestamp())
    .withColumn("source_path", coalesce(col("_metadata.file_path"), lit(SOURCE_CSV)))
    .withColumn("batch_id", lit("batch_002"))
    .withColumn("load_comment", lit("append_with_mergeSchema"))
)
append_batch.write.format("delta").mode("append").option("mergeSchema", "true").save(P_BASIC)
print("After append, total rows:", spark.read.format("delta").load(P_BASIC).count())
spark.read.format("delta").load(P_BASIC).printSchema()

## Part E — JSON → Delta + **partitioned** write (lab *08* style)

In [0]:
from pyspark.sql.functions import col, substring, upper as u

jdf = spark.read.format("json").option("inferSchema", True).load(SOURCE_JSON)
jdf = jdf.withColumn("dest_initial", u(substring(col("DEST_COUNTRY_NAME"), 1, 1)))
jdf.write.format("delta").mode("overwrite").partitionBy("dest_initial").option("overwriteSchema", "true").save(P_PART)
print("Partitioned Delta at:", P_PART)

In [0]:
spark.read.format("delta").load(P_PART).groupBy("dest_initial").count().orderBy("dest_initial").display(30)

In [0]:
# Read only one partition (predicate pushdown)
spark.read.format("delta").load(P_PART).where("dest_initial = 'U'").select("DEST_COUNTRY_NAME", "count").display(10, truncate=False)

## Part F — `DESCRIBE DETAIL` + `DESCRIBE HISTORY`

In [0]:
spark.sql(f"DESCRIBE DETAIL delta.`{P_BASIC}`").display(truncate=False)

In [0]:
spark.sql(f"DESCRIBE HISTORY delta.`{P_BASIC}`").select("version", "timestamp", "operation", "operationMetrics").display(20, truncate=False)

## Part F2 — SQL on Delta path (Spark SQL `delta.\`path\``)

Same physical table as **`spark.read.format('delta').load(P_BASIC)`** — analysts often use **`delta.\`abfss://...\``** in SQL warehouses.

### F2a — Row count

In [0]:
spark.sql(f"SELECT COUNT(*) AS c FROM delta.`{P_BASIC}`").display(truncate=False)

### F2b — Top destinations

In [0]:
spark.sql(f"SELECT DEST_COUNTRY_NAME, SUM(count) AS s FROM delta.`{P_BASIC}` GROUP BY 1 ORDER BY 2 DESC LIMIT 8").display(truncate=False)

### F2c — Min / max count

In [0]:
spark.sql(f"SELECT MIN(count) AS mn, MAX(count) AS mx, AVG(count) AS av FROM delta.`{P_BASIC}`").display(truncate=False)

### F2d — Distinct origins

In [0]:
spark.sql(f"SELECT COUNT(DISTINCT ORIGIN_COUNTRY_NAME) AS origins FROM delta.`{P_BASIC}`").display(truncate=False)

### F2e — Filter + limit

In [0]:
spark.sql(f"SELECT * FROM delta.`{P_BASIC}` WHERE count > 1000 ORDER BY count DESC LIMIT 10").display(truncate=False)

## Part G — Schema **strict** vs **evolution**

In [0]:
from pyspark.sql.types import LongType, StringType, StructField, StructType

strict_schema = StructType(
    [
        StructField("DEST_COUNTRY_NAME", StringType(), True),
        StructField("ORIGIN_COUNTRY_NAME", StringType(), True),
        StructField("count", LongType(), True),
    ]
)
sdf = spark.read.format("csv").option("header", True).schema(strict_schema).load(SOURCE_CSV)
sdf.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(P_STRICT)
print("Strict-schema Delta:", P_STRICT)

In [0]:
# Try to append a row with an EXTRA column — expect failure without mergeSchema
from pyspark.sql import Row

bad = spark.createDataFrame([Row(DEST_COUNTRY_NAME="X", ORIGIN_COUNTRY_NAME="Y", count=1, extra_col="oops")])
try:
    bad.write.format("delta").mode("append").save(P_STRICT)
except Exception as e:
    print("Expected failure (schema mismatch):", type(e).__name__)

In [0]:
# Same append with mergeSchema = true
bad.write.format("delta").mode("append").option("mergeSchema", "true").save(P_STRICT)
spark.read.format("delta").load(P_STRICT).printSchema()

## Part H — Multiple **read patterns** (practice)

### H1 — `format('delta').load`

In [0]:
spark.read.format('delta').load(P_BASIC).select('DEST_COUNTRY_NAME', 'count').limit(5).display()

### H2 — SQL `delta.\`path\``

In [0]:
spark.sql(f'SELECT COUNT(*) AS c FROM delta.`{P_BASIC}`').display()

### H3 — `table` after temp view

In [0]:
spark.read.format('delta').load(P_BASIC).createOrReplaceTempView('t_basic'); spark.sql('SELECT MIN(count), MAX(count) FROM t_basic').display()

### H4 — JSON copy for merge lab later

In [0]:
spark.read.format('json').load(SOURCE_JSON).write.format('delta').mode('overwrite').option('overwriteSchema','true').save(P_JSON); print('OK', P_JSON)

## Part I — `replaceWhere` (conditional overwrite) — discussion + pattern

**Idea:** Overwrite **only** rows matching a predicate (partition pruning). Reduces blast radius vs full `overwrite`.

```python
# Example pattern (uncomment & adapt partition column):
# df.write.format("delta").mode("overwrite").option("replaceWhere", "dest_initial = 'Z'").save(P_PART)
```

Use when you partition by a stable key (date, region). **Wrong predicates** can corrupt data — test in dev first.

### I2 — Hands-on: **`replaceWhere`** on a **single** partition (`P_PART`)

Run **after** Part E. This **overwrites only** rows where `dest_initial = 'U'`. We add a marker column with **`mergeSchema`** so you can see the effect in `DESCRIBE HISTORY`.

In [0]:
from pyspark.sql.functions import col, current_timestamp, lit

slice_df = (
    spark.read.format("delta")
    .load(P_PART)
    .where("dest_initial = 'U'")
    .withColumn("partition_refresh_at", current_timestamp())
    .withColumn("replacewhere_note", lit("slice_U_refreshed"))
)
(
    slice_df.write.format("delta")
    .mode("overwrite")
    .option("replaceWhere", "dest_initial = 'U'")
    .option("mergeSchema", "true")
    .save(P_PART)
)
print("Rows in U slice after replace:", spark.read.format("delta").load(P_PART).where("dest_initial = 'U'").count())
spark.sql(f"DESCRIBE HISTORY delta.`{P_PART}`").select("version", "operation").display(5, truncate=False)

## Part K — Analytics on **`P_BASIC`** (aggregations & distribution)

In [0]:
from pyspark.sql.functions import col

b = spark.read.format("delta").load(P_BASIC)
b.groupBy("DEST_COUNTRY_NAME").agg({"count": "sum"}).orderBy(col("sum(count)").desc()).display(12, truncate=False)
b.selectExpr("percentile_approx(count, 0.5) as median_cnt", "max(count) as max_cnt", "min(count) as min_cnt").display()

In [0]:
from pyspark.sql.functions import col

# Top origin countries by total flights
spark.read.format("delta").load(P_BASIC).groupBy("ORIGIN_COUNTRY_NAME").sum("count").orderBy(col("sum(count)").desc()).display(12, truncate=False)

## Part L — **`P_JSON`** lifecycle + **schema** vs CSV table

In [0]:
# Ensure P_JSON exists (idempotent)
spark.read.format("json").option("inferSchema", True).load(SOURCE_JSON).write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(P_JSON)
spark.read.format("delta").load(P_JSON).printSchema()
spark.read.format("delta").load(P_JSON).display(5, truncate=False)

In [0]:
# Column overlap: JSON vs basic Delta (names only)
cols_j = set(spark.read.format("delta").load(P_JSON).columns)
cols_b = set(spark.read.format("delta").load(P_BASIC).columns)
print("Only in JSON delta:", sorted(cols_j - cols_b))
print("Only in basic delta:", sorted(cols_b - cols_j))
print("Shared:", sorted(cols_j & cols_b))

## Part M — **Incremental** mindset: watermark column (`ingested_at`)

Production loads often track **last successful timestamp** or **batch id**. Here, filter **newer** rows (demo pattern only):

```python
# last_ts = spark.read.format("delta").load(P_BASIC).selectExpr("max(ingested_at)").collect()[0][0]
# incremental = spark.read.format("csv").options(header=True, inferSchema=True).load(SOURCE_CSV).where(col("ingested_at") > lit(last_ts))  # CSV has no ts — real pipelines add file date
```

Your **`ingested_at`** on `P_BASIC` is **load time**, not event time — use it to teach **metadata lineage**, not business cutoffs.

In [0]:
from pyspark.sql.functions import col

mx = spark.read.format("delta").load(P_BASIC).selectExpr("max(ingested_at) as mx").collect()[0]["mx"]
print("Latest ingested_at in P_BASIC:", mx)
spark.read.format("delta").load(P_BASIC).where(col("batch_id").isNotNull()).select("batch_id", "ingested_at", "DEST_COUNTRY_NAME").display(8, truncate=False)

## Part N — **Quality** checks: nulls & duplicate route keys

In [0]:
from pyspark.sql.functions import col, count

b = spark.read.format("delta").load(P_BASIC)
b.select([count(col(c)).alias(c) for c in ["DEST_COUNTRY_NAME", "ORIGIN_COUNTRY_NAME", "count"]]).display()
b.groupBy("DEST_COUNTRY_NAME", "ORIGIN_COUNTRY_NAME").count().where("count > 1").orderBy(col("count").desc()).display(10, truncate=False)

## Part O — **Table properties** & **detail** for multiple tables

In [0]:
for label, path in [("basic", P_BASIC), ("partitioned", P_PART), ("json", P_JSON)]:
    print("===", label, "===")
    spark.sql(f"DESCRIBE DETAIL delta.`{path}`").select("format", "partitionColumns", "numFiles").display(truncate=False)

In [0]:
spark.sql(f"SHOW TBLPROPERTIES delta.`{P_BASIC}`").display(30, truncate=False)